<a href="https://colab.research.google.com/github/Eugeacosta/Proyecto-Final-Entretejiendo-Imaginaci-n-y-Algoritmos/blob/main/Proyecto_Final_Entretejiendo_Imaginaci%C3%B3n_y_Algoritmos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Extracción y Preparación de Datos (Simulación)

El primer paso en nuestro framework LogiGen AI es obtener la información cruda. Para el caso de uso de esta POC, analizaremos el ciclo de vida de la **logística inversa** (retiro de equipos).

Para optimizar los costos de la API y garantizar la viabilidad del proyecto (Fast Prompting), **no** enviaremos la base de datos completa al modelo de lenguaje. En su lugar, realizamos una extracción focalizada mediante SQL desde nuestro Data Warehouse, filtrando únicamente los hitos críticos que definen el cumplimiento del SLA (Fecha de Solicitud vs. Fecha de Finalización), el estado actual y la provincia.

La siguiente query representa la extracción utilizada para obtener el dataset base:

```sql
SELECT
    a.Equipo AS [ID Equipo],
    'DIRECTV' AS [Cliente],
    'Inversa' AS [Servicio],
    TRY_CONVERT(date, CAST(a.Fecha_Solicitud_Pick_Up AS CHAR(8))) AS [Fecha Solicitud],
    TRY_CONVERT(date, CAST(a.Fecha_POD AS CHAR(8))) AS [Fecha Finalización],
    a.Destino_Provincia_Descripcion AS [Provincia],
    c.Estado_Equipo_Descripcion AS [Status Actual],
    d.Motivo_Descripcion AS [Motivo Final]
FROM DW.Fact_Visitas_Retiros_Nuevo a
LEFT JOIN dw.Dim_Estados_Equipo c ON a.Estado_Key = c.Estado_Equipo_Key
LEFT JOIN dw.Dim_Motivos d ON a.Motivo_Final_Key = d.Motivo_Key
WHERE
    a.AP = '40041180'
    AND TRY_CONVERT(date, CAST(a.Fecha_Solicitud_Retiro AS CHAR(8))) >= DATEADD(year, -2, GETDATE())
    AND c.Estado_Equipo_Descripcion NOT IN ('AGUARDANDO PROCESO', 'CIERRE OPERATIVO');


In [2]:
!pip install gradio gtts google-genai

In [3]:
!pip install google-genai

In [4]:
import gradio as gr
from google import genai
from google.colab import userdata
from gtts import gTTS
import os
gemini_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=gemini_key)


In [5]:
# 2. Función principal que hará todo el trabajo (Texto y Audio)
def procesar_logistica(datos_ingresados):
    # A. Generar el reporte con Gemini (Texto-Texto)
    prompt = f"""
    Actúa como un Senior Data Analyst especializado en Supply Chain.
    Analiza este reporte de logística inversa:
    ###
    {datos_ingresados}
    ###
    Diagnostica el SLA (Verde/Amarillo/Rojo), enumera 2 causas raíz y da 1 sugerencia breve.
    """

    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt
        )
        reporte_texto = response.text

        # B. Generar la alerta en Audio (Texto-Audio)
        # Extraemos solo la primera línea del reporte para el audio (el diagnóstico)
        texto_audio = "Atención equipo de logística. " + reporte_texto.split('\n')[0]

        # Creamos el archivo de audio con gTTS
        tts = gTTS(text=texto_audio, lang='es', tld='com.ar') # Acento argentino
        ruta_audio = "alerta_sla.mp3"
        tts.save(ruta_audio)

        return reporte_texto, ruta_audio

    except Exception as e:
        return f"Error: {e}", None

# 3. Datos por defecto para que el usuario no tenga que escribir todo de cero
datos_ejemplo = """Cliente: DIRECTV
Volumen: 3,250
SLA Cumplido: 82% (Meta 95%)
Provincia crítica: Buenos Aires
Motivo principal: Domicilio Cerrado"""

# 4. Creación de la Interfaz Visual con Gradio
interfaz = gr.Interface(
    fn=procesar_logistica, # La función que armamos arriba
    inputs=gr.Textbox(lines=6, value=datos_ejemplo, label="Ingresar Datos Logísticos (KPIs semanales)"),
    outputs=[
        gr.Markdown(label="Reporte Gerencial Generado (IA)"), # Salida 1: Texto
        gr.Audio(label="Alerta Sonora (Texto a Audio)")       # Salida 2: Audio
    ],
    title="LogiGen AI: Framework de Análisis de SLAs",
    description="Ingresa las métricas semanales. La IA generará un diagnóstico de Causa-Raíz y una alerta en audio para el centro de distribución.",
    theme="soft"
)

# 5. Lanzar la aplicación
interfaz.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://51c28baf49a513cda4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Created dataset file at: .gradio/flagged/dataset1.csv
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://51c28baf49a513cda4.gradio.live
